In [ ]:
from langchain_core.caches import InMemoryCache
from langgraph.types import CachePolicy
from dataclasses import dataclass

from langchain.chat_models import init_chat_model
from langchain.messages import SystemMessage
from langgraph.graph import END,MessagesState,StateGraph, START
from langgraph.runtime import Runtime
from typing_extensions import TypedDict
from langgraph.types import RetryPolicy
from dotenv import load_dotenv
load_dotenv()
import os

os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

@dataclass
class ContextSchema:
    model_provider:str = "anthropic"
    system_message:str | None = None

MODELS = {
    "anthropic": init_chat_model("claude-haiku-4-5-20251001"),
    "openai": init_chat_model("gpt-5-mini",model_provider="openai")
}

def call_model(state:MessagesState,runtime:Runtime[ContextSchema]):
    model = MODELS[runtime.context.model_provider]
    messages = state["messages"]
    if(system_message := runtime.context.system_message):
        messages = [SystemMessage(system_message)] + messages
    response = model.invoke(messages)
    return {"messages":[response]}

builder = StateGraph(MessagesState,context_schema=ContextSchema)
builder.add_node("model",call_model,retry_policy=RetryPolicy(max_attempts=3, initial_interval=1.0),cache_policy=CachePolicy(ttl=120))

# builder.add_edge(START,"model")
# builder.add_edge("model",END)

builder.add_sequence([model])
builder.add_edge(START,"model")

graph = builder.compile()
# graph = builder.compile(cache=InMemoryCache())

# Usage
input_message = {"role":"user","content": "Hi,My name is Deepak. I am an automation test engieer. I am exploring generative AI in testing."}
# response = graph.invoke({"messages":[input_message]},context = {"model_provider":"openai","system_message":"respond in french"})
response = graph.invoke({"messages":[input_message]},context = {"model_provider":"anthropic","system_message":"respond in english"})

print(f"response: {response}")

for message in response["messages"]:
    message.pretty_print()

response: {'messages': [HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}, id='a962f2c1-0a76-4f4b-92d9-59166e3ce639'), AIMessage(content='Hello! How can I help you today?', additional_kwargs={}, response_metadata={'id': 'msg_01L3Zv6Frj2BKrMZrfx29Rfc', 'model': 'claude-haiku-4-5-20251001', 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'input_tokens': 11, 'output_tokens': 12, 'server_tool_use': None, 'service_tier': 'standard', 'inference_geo': 'not_available'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'}, id='lc_run--019c7f92-5f72-7563-8efa-cac2b8b02667-0', usage_metadata={'input_tokens': 11, 'output_tokens': 12, 'total_tokens': 23, 'input_token_details': {'cache_read': 0, 'cache_creation': 0, 'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}})]}
==================